# Nextbike Brno Availability Baseline

Generated at `2026-05-25T10:00:13+02:00`.

This notebook reproduces the first 30-minute station emptiness baseline/model analysis.


## Current Summary

| Metric | Value |
|---|---|
| `collection_runs` | `809` |
| `first_collected_at` | `2026-05-24 18:32:12+02:00` |
| `latest_collected_at` | `2026-05-25 09:59:57+02:00` |
| `station_rows` | `240273` |
| `free_bike_rows` | `443058` |
| `distinct_stations` | `297` |
| `bikes_available_min_avg_max` | `542/564.22/573` |
| `db_size_mb` | `23.01` |

## Dataset Summary

| Metric | Value |
|---|---|
| `rows` | `229581` |
| `stations` | `297` |
| `first_collected_at` | `2026-05-24 20:17:28+02:00` |
| `latest_collected_at` | `2026-05-25 09:29:04+02:00` |
| `empty_future_rate` | `0.3982` |
| `avg_abs_change` | `0.0888` |
| `changed_rate` | `0.0707` |
| `null_lag_5m` | `2970` |
| `null_lag_15m` | `6831` |

Build settings:

- horizon minutes: `30`
- max target delay minutes: `40`
- lag tolerance minutes: `10`


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'nextbike.duckdb').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'nextbike.duckdb'
TABLE_NAME = 'training_station_availability'

con = duckdb.connect(str(DB_PATH), read_only=True)
df = con.sql(f"select * from {TABLE_NAME} order by collected_at, station_id").df()
df.head()

In [ ]:
df[['bikes_now', 'bikes_future', 'empty_now', 'empty_future']].describe(include='all')

## Baseline Metrics

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | Avg precision | Pred empty rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| `persistence_empty_now` | 0.9581 | 0.9589 | 0.9408 | 0.9498 | 0.9557 | 0.9271 | 0.4135 |
| `station_prior_empty_rate` | 0.8675 | 0.8764 | 0.7981 | 0.8354 | 0.9107 | 0.8792 | 0.3838 |
| `low_inventory_le_1` | 0.8116 | 0.6937 | 0.9900 | 0.8158 | 0.9711 | 0.9405 | 0.6015 |
| `majority_train_class` | 0.5785 | 0.0000 | 0.0000 | 0.0000 | 0.5000 | 0.4215 | 0.0000 |

In [ ]:
from nextbike_analysis.modeling import evaluate_baselines

baseline = evaluate_baselines(
    db_path=DB_PATH,
    table_name=TABLE_NAME,
    test_fraction=0.25,
    low_bike_threshold=1,
)
[(row.name, row.f1, row.roc_auc) for row in baseline.metrics]

## Model Metrics

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | Avg precision | Pred empty rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| `hist_gradient_boosting` | 0.9548 | 0.9547 | 0.9372 | 0.9459 | 0.9814 | 0.9728 | 0.4137 |
| `logistic_regression` | 0.9400 | 0.9283 | 0.9295 | 0.9289 | 0.9783 | 0.9685 | 0.4220 |

In [ ]:
from nextbike_analysis.modeling import train_models

model_result, _ = train_models(
    db_path=DB_PATH,
    table_name=TABLE_NAME,
    test_fraction=0.25,
    model_dir=None,
)
[(row.name, row.f1, row.roc_auc) for row in model_result.metrics]

## Recommendation Metrics

| Strategy | Success rate | Avg dist m | P90 dist m | No candidate | Avg extra m | Avg empty probability |
|---|---:|---:|---:|---:|---:|---:|
| `bikes>=2` | 0.9979 | 161 | 259 | 0 | 29 | 0.0029 |
| `dist+risk` | 0.9113 | 137 | 222 | 0 | 5 | 0.0113 |
| `model<0.4` | 0.8876 | 135 | 222 | 0 | 3 | 0.0298 |
| `nearest` | 0.8649 | 132 | 222 | 0 | 0 | 0.1220 |

In [ ]:
from nextbike_analysis.recommendations import evaluate_recommendation_strategies

recommendation_result = evaluate_recommendation_strategies(
    db_path=DB_PATH,
    model_path=ROOT / 'data' / 'models' / 'hist_gradient_boosting.joblib',
    table_name=TABLE_NAME,
    test_fraction=0.25,
    max_distance_m=1000.0,
)
[(row.label, row.success_rate, row.avg_distance_m) for row in recommendation_result.metrics]

## Notes

The current sample is mostly overnight, so persistence is a very strong baseline.
Do not judge advanced model classes until we have morning/daytime demand in the dataset.
